# Generate Complete corpus papers origen-definitivos

1. **Output Directory:** Writes to `papers-origen/latex-edit/`, preserving `latex-files/`.
2. **Optimized Cleaning:** Removed the per-file cleaning step. Instead, after all files are processed (Split + Replace), a **single global `sed` pass** is applied to the destination folder.
3. **Logic Order:**
   - Step 1: Extract mappings & generate `indice-*.csv`.
   - Step 2: **Split** grouped citations.
   - Step 3: **Replace** keys with `arXiv:...`.
   - **Final Step:** Apply global `sed` to protect `arXiv:` and empty all others.

**Global SED Logic:**
`\cite{arXiv:...}` → Protected → Restored
`\cite{anything_else}` → `\cite{}`

## 1. Configuration

In [ ]:
%%bash
cat > ./cite_config.env <<'EOF'
SOURCE_FOLDER=./papers-origen/latex-files     # Source folder (Original extracted files)
PAPERS_EDIT_FOLDER=./papers-origen/latex-edit # Destination folder 
SEARCH_DEPTH=1                                # Search depth
MATCH_CITES_DIR=./cite_work_temp              # Temporary directory
INDICES_FOLDER=./indices_output               # Output folder for individual indices
SCRIPT_OPTS=""

CITES_FOLDER=./papers-origen/cites
PAPERS_DEFINITIVOS=./papers-origen-definitivos
EOF

source ./cite_config.env
echo "Config loaded:"
echo "  SOURCE             : $SOURCE_FOLDER"
echo "  EDIT LATEX         : $PAPERS_EDIT_FOLDER"
echo "  INDICES            : $INDICES_FOLDER"

## 2. Verify Input Files

In [ ]:
%%bash
source cite_config.env

if [ ! -d "$SOURCE_FOLDER" ]; then
  echo "ERROR: Source folder not found: $SOURCE_FOLDER"
  exit 1
fi

# Prepare destination
rm -rf "$PAPERS_EDIT_FOLDER"
mkdir -p "$PAPERS_EDIT_FOLDER"

count=$(find "$SOURCE_FOLDER" -maxdepth "$SEARCH_DEPTH" -type f -iname '*.tex' | wc -l)
echo "Found $count .tex files in $SOURCE_FOLDER"
echo "Destination prepared: $PAPERS_EDIT_FOLDER"

# [ "$count" -eq 0 ] && echo "ERROR: No files found." && exit 1

## 3. Process: Extract, Split, Replace

Executes the first 3 steps on copies of the source files. The final cleaning will happen in the next cell.

In [ ]:
%%bash
set -uo pipefail
source cite_config.env

# Setup directories
mkdir -p "$INDICES_FOLDER"
rm -rf "$MATCH_CITES_DIR" && mkdir -p "$MATCH_CITES_DIR"

# Regex patterns
NUEVO='[0-9]{4}\.[0-9]{4,5}'
VIEJO='(^|[^A-Za-z0-9.-])(astro-ph|cond-mat|gr-qc|hep-ex|hep-lat|hep-ph|hep-th|math-ph|nlin|nucl-ex|nucl-th|physics|quant-ph|math|cs|q-bio|q-fin|stat|chao-dyn|solv-int|alg-geom|dg-ga|funct-an|q-alg|patt-sol|adap-org|cmp-lg|comp-gas|mtrl-th|supr-con|chem-ph|atom-ph|bayes-an|plasm-ph|acc-phys|ao-sci)(\.[A-Z]{2})?/[0-9]{7}'
AX='[Aa][Rr][Xx][Ii][Vv]'
ID='[0-9]{4}\.[0-9]{4,5}(v[0-9]+)?'
RE_SI="$AX.*$NUEVO|$NUEVO.*$AX|$VIEJO"

# --- Helpers ---
limpiar() { sed -E 's/\\(url|href|nolinkurl)//g; s/[{}]//g; s/~/ /g; s/[[:space:]]+/ /g; s/^ //; s/ $//'; }

solo_id() {
  local t id
  t=$(cat)
  id=$(printf '%s\n' "$t" | grep -oE "$AX[^0-9]*$ID" | head -n 1 | grep -oE "$ID")
  [[ -z $id ]] && id=$(printf '%s\n' "$t" | grep -oE "$VIEJO" | head -n 1 | sed -E 's/^[^A-Za-z]//')
  [[ -z $id ]] && printf '%s\n' "$t" | grep -qE "$RE_SI" && id=$(printf '%s\n' "$t" | grep -oE "$ID" | head -n 1)
  [[ -n $id ]] && printf 'arXiv:%s\n' "$id"
}

una_linea() {
  if [[ $1 == *.bib ]]; then tr '\r\n' '  ' < "$1"
  else grep -v '^[[:space:]]*%' "$1" | tr '\r\n' '  '; fi
  echo
}

leer_bibitems() {
  local sep=$'\001'
  una_linea "$1" \
  | sed -e "s/\\\\end{thebibliography}/$sep/g" -e "s/\\\\bibitem/$sep@BIBITEM@/g" | tr "$sep" '\n' \
  | sed -nE "s/^@BIBITEM@[[:space:]]*(\[[^]]*\])?[[:space:]]*\{[[:space:]]*([^}]*[^}[:space:]])[[:space:]]*\}(.*)$/\2\t\3/p" \
  | while IFS= read -r linea; do
      clave=${linea%%$'\t'*}; texto=${linea#*$'\t'}
      seg=$(printf '%s\n' "$texto" | limpiar | solo_id)
      [[ -n $seg ]] && printf '%s\t%s\n' "$clave" "$seg"
    done
}

leer_entradas() {
  local sep=$'\001'
  una_linea "$1" \
  | sed -E -e "s/(^|[[:space:]])@([A-Za-z]+)[[:space:]]*\{/\1$sep@\2{/g" \
           -e "s/\\\\end\{filecontents\*?\}/$sep/g" | tr "$sep" '\n' \
  | grep -E '^@[A-Za-z]+\{' | grep -viE '^@(comment|string|preamble)\{' \
  | while IFS= read -r e; do
      clave=$(printf '%s\n' "$e" | sed -nE 's/^@[A-Za-z]+\{[[:space:]]*([^,[:space:]]+)[[:space:]]*,.*/\1/p')
      [[ -n $clave ]] || continue
      campos=$(printf '%s\n' "$e" \
        | sed -E "s/,[[:space:]]*([A-Za-z][A-Za-z_-]*)[[:space:]]*=[[:space:]]*/$sep\1=/g" \
        | tr "$sep" '\n' | sed '1d' \
        | sed -E 's/^([^=]*)="(.*)"[[:space:]}]*$/\1=\2/' | limpiar | sed -E 's/[[:space:],]+$//')
      valor=""
      cand=$(printf '%s\n' "$campos" | grep -E "$RE_SI")
      if [[ -n $cand ]]; then
        for n in journal journaltitle booktitle publisher howpublished note eprint url doi; do
          valor=$(printf '%s\n' "$cand" | grep -i -m1 "^$n=")
          [[ -n $valor ]] && break
        done
        [[ -n $valor ]] || valor=$(printf '%s\n' "$cand" | head -n 1)
        valor=${valor#*=}
      elif printf '%s\n' "$campos" | grep -iE '^(archiveprefix|eprinttype)=' | grep -qi arxiv; then
        ep=$(printf '%s\n' "$campos" | grep -i -m1 '^eprint=')
        ep=${ep#*=}
        printf '%s\n' "$ep" | grep -qE "$NUEVO|$VIEJO" && valor="arXiv:$ep"
      fi
      [[ -n $valor ]] && valor=$(printf '%s\n' "$valor" | solo_id)
      [[ -n $valor ]] && printf '%s\t%s\n' "$clave" "$valor"
    done
}

extraer_equivalencias() {
  local f=$1 crudo=$MATCH_CITES_DIR/crudo_$$_tmp dir nombre linea k v vistos=$'\t'
  : > "$crudo"
  leer_bibitems "$f" >> "$crudo"
  leer_entradas "$f" >> "$crudo"
  dir=$(dirname "$f")
  while IFS= read -r nombre; do
    [[ $nombre == /* ]] || nombre=$dir/$nombre
    [[ -f $nombre ]] && leer_entradas "$nombre" >> "$crudo"
  done < <(grep -v '^[[:space:]]*%' "$f" \
           | grep -oE '\\(bibliography|addbibresource)[[:space:]]*(\[[^]]*\])?\{[^}]*\}' \
           | sed -E 's/.*\{([^}]*)\}$/\1/' | tr ',' '\n' \
           | sed -E 's/^[[:space:]]+//; s/[[:space:]]+$//; /^$/d; /\.bib$/!s/$/.bib/')
  while IFS= read -r linea; do
    linea=${linea#$'\t'}
    k=${linea%%$'\t'*}; v=${linea#*$'\t'}
    case $vistos in *"$'\t'$k$'\t'"*) continue ;; esac
    vistos+="$k$'\t'"
    [[ -n $v ]] || v=$(grep -F "$'\t'$k$'\t'" "$crudo" | grep -v "$'\t'$" | head -n 1 | cut -f3-)
    [[ -n $v ]] && printf '%s\t%s\n' "$k" "$v"
  done < "$crudo"
  rm -f "$crudo"
}

# --- SED Programs ---

# 1. REPLACE: Generates dynamic sed script to change key -> arXivID
generar_sed_reemplazo() {
  local n=0 k v rk rv
  while IFS=$'\t' read -r k v; do
    n=$(( n + 1 ))
    rk=$(printf '%s' "$k" | sed 's/[][\.*^$+?(){}|/]/\\&/g')
    rv=$(printf '%s' "$v" | sed 's/[\/&]/\\&/g')
    printf ':c%d\n' "$n"
    printf 's/(\\\\[A-Za-z]*cite[A-Za-z]*\*?([[:space:]]*\\[[^]]*\\])*[[:space:]]*\{([^}]*[,[:space:]])?)%s([[:space:]]*[,}])/\\1%s\\4/\n' "$rk" "$rv"
    printf 't c%d\n' "$n"
  done
}

# 2. SPLIT: Separates lists \cite{a,b} -> \cite{a}, \cite{b}
SED_SPLIT_CMD=':split; s/(\\[Cc]ite[a-z]*\*?)([[:space:]]*\[[^]]*\])?([[:space:]]*\{[[:space:]]*)([^},]+[^},[:space:]])([[:space:]]*,[[:space:]]*)([^}]*\})/\1\2\3\4}, \1{\6/g; t split'

# --- Main Loop ---
total=0 ok_count=0 no_map_count=0

while IFS= read -r -d '' src_file; do
  total=$((total+1))
  basename_full=$(basename "$src_file")
  base="${basename_full%.tex}"
  
  # Define paths
  dest_file="$PAPERS_EDIT_FOLDER/$basename_full"
  tabla="$MATCH_CITES_DIR/tabla_$$.tsv"
  indice_out="$INDICES_FOLDER/indice-$base.csv"
  prog_replace="$MATCH_CITES_DIR/prog_replace_$$.sed"
  
  # Copy source to destination first
  cp "$src_file" "$dest_file"
  
  # 1. Extract mappings & Write Index
  extraer_equivalencias "$dest_file" > "$tabla"
  
  # Robust count to prevent syntax errors
  n_keys=$(grep -c . "$tabla" 2>/dev/null | tr -d '[:space:]')
  [[ -z "$n_keys" ]] && n_keys=0
  
  : > "$indice_out"
  if (( n_keys > 0 )); then
    while IFS=$'\t' read -r k v; do
      printf '"%s","%s"\n' "${k//\"/\"\"}" "${v//\"/\"\"}" >> "$indice_out"
    done < "$tabla"
    generar_sed_reemplazo < "$tabla" > "$prog_replace"
  fi
  
  # 2. Apply Transformations (Split + Replace)
  ok=1
  
  # STEP A: SPLIT all groups first
  sed -i -E "$SED_SPLIT_CMD" "$dest_file" || ok=0
  
  # STEP B: REPLACE keys with arXiv IDs
  if (( n_keys > 0 )); then
    sed -i -E -f "$prog_replace" "$dest_file" || ok=0
  fi
  
  # NOTE: STEP C (Clean) is REMOVED from here. It will run globally at the end.
  
  if (( ok == 0 )); then
    echo "[ERROR] $basename_full"
  else
    if (( n_keys > 0 )); then
      echo "[OK] $basename_full -> $n_keys replacements made. Index: indice-$base.csv"
      ok_count=$((ok_count+1))
    else
      echo "[INFO] $basename_full -> No mappings found (will be cleaned globally later)."
      no_map_count=$((no_map_count+1))
    fi
  fi
  
  rm -f "$tabla" "$prog_replace"
done < <(find "$SOURCE_FOLDER" -maxdepth "$SEARCH_DEPTH" -type f -iname '*.tex' -print0 | LC_ALL=C sort -z)

echo
echo "=== Phase 1 Complete ==="
echo "Processed: $total"
echo "With replacements: $ok_count"
echo "No mappings: $no_map_count"
echo "Indices saved in: $INDICES_FOLDER"
echo "Files ready for global clean in: $PAPERS_EDIT_FOLDER"

## 4. Final Global Clean

Applies the optimized `sed` command to the entire `latex-edit/` folder to:
1. Protect `\cite{arXiv:...}`
2. Empty all other `\cite{...}` to `\cite{}`
3. Restore the protected ones.

In [ ]:
%%bash
source cite_config.env

if [ ! -d "$PAPERS_EDIT_FOLDER" ]; then
  echo "ERROR: Destination folder not found. Run Phase 1 first."
  exit 1
fi

cd "$PAPERS_EDIT_FOLDER" || exit 1

echo "Applying global citation cleanup to $(ls *.tex | wc -l) files..."

# The user-provided optimized SED command
# 1. Mark arXiv cites with PROTEGE_CITAS_ARXIV
# 2. Empty ALL other cites
# 3. Restore the mark to backslash
sed -E -i 's/\\([Cc]ite[a-z]*\*?[[:space:]]*(\[[^]]*\][[:space:]]*)*\{[[:space:]]*arXiv:)/PROTEGE_CITAS_ARXIV\1/g; s/\\[Cc]ite[a-z]*\*?[[:space:]]*(\[[^]]*\][[:space:]]*)*\{[^}]*\}/\\cite{}/g; s/PROTEGE_CITAS_ARXIV/\\/g' *.tex

if [ $? -eq 0 ]; then
  echo "SUCCESS: Global cleanup completed."
  echo "All non-arXiv citations are now empty (\cite{})."
  echo "All arXiv citations are preserved."
else
  echo "ERROR: Global cleanup failed."
  exit 1
fi

## 7. Paso 4: Extracción de citas arXiv normalizadas
Busca todas las ocurrencias de comandos `\cite` que contengan un identificador `arXiv:<id>` dentro de los archivos editados (`edit/*-edit.tex`) y consolida el listado resultante en `cites/citas-definitivas-1L.txt`.

In [ ]:
%%bash
set -euo pipefail
source cite_config.env

mkdir -p "$CITES_FOLDER"

OUT_CITES="citas-definitivas.txt"
OUT_DEFINITIVOS="papers-origen-definitivos.txt"

echo "$CITES_FOLDER"
echo "$PAPERS_EDIT_FOLDER"
cd   "$PAPERS_EDIT_FOLDER"

grep -P -i 'cite.?\{[^}]*arXiv:\d+\.\d+[^}]*\}' *.tex > "../../$CITES_FOLDER/$OUT_CITES" || true

total_lineas=$(wc -l < "../../$CITES_FOLDER/$OUT_CITES" | tr -d ' ')
echo "Citas extraídas guardadas en: $OUT_CITES ($total_lineas coincidencias)"    

In [ ]:
%%bash
source cite_config.env

cd "$CITES_FOLDER" && pwd

OUT_CITES="citas-definitivas.txt"
OUT_DEFINITIVOS="papers-origen-definitivos.txt"

# cut -d: -f1 citas-definitivas.txt | sort -u > papers-definitivos.txt
cut -d: -f1 "$OUT_CITES" | sort -u > "$OUT_DEFINITIVOS"

total_definitivos=$(wc -l < "../../$CITES_FOLDER/$OUT_DEFINITIVOS" | tr -d ' ')
echo "Archivos con citas arXiv (en Latex): $OUT_CITES ($total_definitivos papers definitivos con citas arXiv válidas)" 

In [ ]:
%%bash
source cite_config.env

echo "$PAPERS_DEFINITIVOS"
cd   "$CITES_FOLDER/.."
mkdir -p "$PAPERS_DEFINITIVOS"
cd  "../$CITES_FOLDER"
pwd

rsync -av --files-from=papers-origen-definitivos.txt ../latex-edit/ ../papers-origen-definitivos/

> Fin del notebook 